# Phần 1 — PASTIS và nhãn

**Mục tiêu:** trên Drive có:
- `processed/`: `meta.csv`, `labels.npy`, `instances.npy`, `parcel_ids.npy`, `splits.json`, `dates_s2.json`, `class_pixel_counts.csv`, `selection_seed42.json`.
- `pastis_subset/`: khoảng 45 patch (25 patch của RF pilot cộng 20 patch dùng cho QA), để các phần sau không cần file zip 28,8 GB nữa.

**Runtime:** Runtime → Change runtime type → **CPU**.
**Thời gian:** 30–70 phút, chủ yếu là tải PASTIS.
**Drive cần cho phần này:** khoảng 1–1,5 GB (ước lượng); cộng thêm 28,8 GB nếu chọn giữ zip.

**Code cần có trên GitHub trước khi chạy:**
- `requirements.txt`, `.gitignore`
- `src/__init__.py`, `src/data/__init__.py`, `src/data/pastis_grid.py`, `src/data/pastis_zip.py`, `src/data/selection.py`
- `scripts/build_meta_labels.py`, `scripts/extract_subset.py`
- `notebooks/data/01_pastis_labels.ipynb` (chính notebook này)

## 1. Runtime và Drive
**Xong khi:** Python ≥ 3.12 (geotessera yêu cầu) và `/content` còn trống ≥ 40 GB.

In [ ]:
import sys, shutil, platform
print("Python", platform.python_version())
assert sys.version_info >= (3, 12), "geotessera cần Python >= 3.12"
print(f"/content trống: {shutil.disk_usage('/content').free / 1e9:.0f} GB")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!df -h /content/drive | tail -1

**Cấu hình.** Nếu `pj_vmi` là thư mục bạn cùng nhóm chia sẻ: trên drive.google.com, chuột phải vào thư mục → Organize → Add shortcut → My Drive. Khi đó đường dẫn trong Colab là `/content/drive/MyDrive/pj_vmi` (hoặc đúng tên shortcut bạn đặt).

Lưu ý: file bạn tạo trong thư mục người khác chia sẻ vẫn thuộc về bạn, nên tính vào dung lượng Drive **của bạn**.

Sửa `BRANCH` thành nhánh mà bạn đã push code lên.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"
CODE = Path("/content/code")

ZIP = Path("/content/PASTIS.zip")
ZIP_DRIVE = DRIVE / "raw" / "PASTIS.zip"
KEEP_ZIP_ON_DRIVE = False          # True nếu Drive còn >= 30 GB: lần sau khỏi tải lại
PROC = Path("/content/processed")   # làm trên ổ local (nhanh) rồi copy lên Drive
PROC_DRIVE = DRIVE / "processed"
SUBSET = DRIVE / "pastis_subset"

for p in [PROC, PROC_DRIVE]:
    p.mkdir(parents=True, exist_ok=True)

## 2. Lấy code
Lần đầu thì clone; các lần sau chỉ pull. Ô này in commit đang dùng: hãy ghi nó vào biên bản cùng với kết quả.

**Lỗi thường gặp:**
- Báo `Remote branch ... not found`: sai tên `BRANCH`.
- Thiếu file khi chạy các ô sau: bạn chưa push đủ các file liệt kê ở đầu notebook.

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt
!python -c "import geotessera, geopandas; print('geotessera', geotessera.__version__, '| geopandas', geopandas.__version__)"

## 3. Tải PASTIS
Nguồn: Zenodo record 5012942, file `PASTIS.zip`, 28.760.245.504 byte, md5 `cfc441bf18137ff0bbf4fad58828fb98` (lấy từ API Zenodo ngày 26/10/2026), giấy phép CC-BY 4.0.

**Không giải nén:** các script đọc thẳng trong zip (giải nén ra sẽ tốn 39,5 GB).

**Lỗi thường gặp:** Zenodo chậm hoặc đứt giữa chừng → chạy lại ô tải, `wget -c` sẽ tải tiếp. Nếu runtime bị reset thì `/content` mất hết, phải tải lại (trừ khi đã giữ zip trên Drive).

In [ ]:
URL = "https://zenodo.org/api/records/5012942/files/PASTIS.zip/content"
SIZE, MD5 = 28_760_245_504, "cfc441bf18137ff0bbf4fad58828fb98"

if ZIP.exists() and ZIP.stat().st_size == SIZE:
    print("PASTIS.zip đã đủ, bỏ qua")
elif ZIP_DRIVE.exists():
    print("khôi phục từ Drive..."); shutil.copy(ZIP_DRIVE, ZIP)
else:
    get_ipython().system(f'wget -c -q --show-progress -O "{ZIP}" "{URL}"')
print("kích thước đúng:", ZIP.stat().st_size == SIZE)

In [ ]:
out = !md5sum "{ZIP}"
print(out[0])
assert out[0].split()[0] == MD5, "md5 sai: xóa file rồi chạy lại ô tải"
if KEEP_ZIP_ON_DRIVE and not ZIP_DRIVE.exists():
    ZIP_DRIVE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(ZIP, ZIP_DRIVE)   # khoảng 10–30 phút

## 4. meta.csv và nhãn
Lưới pixel của PASTIS là UTM của từng tile, không phải Lambert-93. Góc trên-trái của mỗi patch được làm tròn về bội số 10 m gần nhất. Góc chưa làm tròn (góc mà code của bạn cùng nhóm dùng) vẫn được giữ trong `x0_raw` / `y1_raw`.

Zip có 2.468 file S2 nhưng chỉ 2.433 patch có nhãn. 35 file S2 thừa không có trong `metadata.geojson` và được bỏ qua.

**Xong khi** (các số đã đo trên dữ liệu thật ngày 26/10):
- 2.433 patch.
- Fold {1: 487, 2: 494, 3: 474, 4: 482, 5: 496}.
- Tile {t30uxv: 531, t31tfj: 623, t31tfm: 723, t32ulu: 556}.
- S2 là `int16`, mọi patch có số ngày khớp với metadata.
- Không có dòng `WARNING ... footprints`.
- Phần làm tròn không đổi trong mỗi tile: t30uxv (−2,36; −0,62), t31tfj (0,90; −3,51), t31tfm (1,84; 0,05), t32ulu (−2,27; −0,49) mét.

In [ ]:
!python scripts/build_meta_labels.py --pastis "{ZIP}" --out "{PROC}"

In [ ]:
meta = pd.read_csv(PROC / "meta.csv", index_col="idx")
labels = np.load(PROC / "labels.npy", mmap_mode="r")
assert len(meta) == 2433 and labels.shape == (2433, 128, 128)
print("footprint_ok:", meta.footprint_ok.value_counts().to_dict())
print(meta.groupby("TILE")[["snap_dx", "snap_dy"]].agg(["min", "max"]).round(2))
# phân bố pixel theo lớp và fold: so với Hình 3 trong tài liệu PASTIS
display(pd.read_csv(PROC / "class_pixel_counts.csv", index_col=0).T)

**25 patch của RF pilot.** Danh sách này phải trùng với `selection.json` của `prepare_batch.py`. Nếu file đó có trên Drive ở `pj_vmi/rf_pilot/batch_2019/selection.json`, ô dưới sẽ tự so sánh; nếu không, hãy so bằng mắt.

In [ ]:
sel = json.loads((PROC / "selection_seed42.json").read_text())
mine = [(p["patch_id"], p["fold"]) for p in sel["patches"]]
print(mine)
theirs_file = DRIVE / "rf_pilot" / "batch_2019" / "selection.json"
if theirs_file.exists():
    theirs = [(p["patch_id"], p["fold"]) for p in json.loads(theirs_file.read_text())["patches"]]
    print("trùng với selection.json của bạn cùng nhóm:", mine == theirs)

In [ ]:
small = ["meta.csv", "labels.npy", "instances.npy", "parcel_ids.npy", "splits.json", "dates_s2.json",
         "class_pixel_counts.csv", "selection_seed42.json"]
for f in small:
    shutil.copy(PROC / f, PROC_DRIVE / f)
    print(f"{f:24s} {(PROC_DRIVE / f).stat().st_size / 1e6:8.1f} MB")

## 5. Tập con PASTIS cho các phần sau
Script chép từ zip lên Drive các file của khoảng 45 patch: `TARGET`, `ParcelIDs`, `INSTANCES`, `S2`, cùng `metadata.geojson`. Thư mục này giữ đúng cấu trúc của PASTIS, nên script của bạn cùng nhóm và các phần 2–4 đọc được như bộ dữ liệu đầy đủ.

**Xong khi:** dòng cuối in ra số patch (`selection_seed42: 25, qa: 20`) và dung lượng (khoảng 0,5–1 GB).

In [ ]:
!python scripts/extract_subset.py --pastis "{ZIP}" --processed "{PROC}" --out "{SUBSET}"

## Gửi lại cho tôi
1. Dòng commit in ra ở bước 2.
2. Toàn bộ output của ô build ở bước 4, cùng kết quả ô kiểm tra ngay sau đó.
3. Danh sách 25 patch, và kết quả so với `selection.json` của bạn cùng nhóm (nếu có).
4. Dòng cuối của bước 5.
5. Nếu ô nào báo lỗi: gửi nguyên văn thông báo lỗi.

Sau đó mới sang phần 2 (TESSERA).